In [ ]:
import networkx as nx
import os
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
import matplotlib.patches as patches
import matplotlib.font_manager
from scipy.stats import pearsonr
from scipy.stats import linregress
from matplotlib import pyplot as plt
import matplotlib as mpl
from pycirclize import Circos
matplotlib.font_manager.fontManager.addfont('/h/tianyi/TS_datasets_reversion/Cell_Press_plot/Arial.ttf')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
plt.rcParams['font.family'] = 'Arial'
mpl.rcParams['font.size'] = 8 
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['text.color'] = 'black'
plt.rcParams['axes.labelcolor'] = 'black'
plt.rcParams['xtick.color'] = 'black'
plt.rcParams['ytick.color'] = 'black'
plt.rcParams['axes.titlecolor'] = 'black'
plt.rcParams['legend.labelcolor'] = 'black'
plt.rcParams['axes.linewidth'] = 0.5

## Figure3A

In [ ]:
## sa ts mut subnetwork number

net_stas_all=pd.read_csv('/sa_ts_mut_subnetwork_number_v2.txt',sep='\t')
tissue_color=pd.read_csv('/tissue_color.txt',sep='\t')
exp_mut_tissue=pd.read_csv('/mutation_expressed_classification_number.txt',sep='\t')
exp_tissue=exp_mut_tissue[exp_mut_tissue['Group']=='Tissue-specific expressed genes']
exp_tissue.drop(['Group'],axis=1,inplace=True)
exp_tissue.columns=['Tissue','Tissue-specific expressed genes']
net_stas_all_addexp=pd.merge(net_stas_all,exp_tissue,on='Tissue')
net_stas_all_addexp['Number']=net_stas_all_addexp['Tissue-specific mutated genes']+net_stas_all_addexp['Tissue-specific subnetworks']+net_stas_all_addexp['Tissue-specific expressed genes']

tissue_type_name_process=tissue_color[['tissue','tissue_name']]
net_stas_all['Tissue']=net_stas_all['Tissue'].str.replace(' ','_')
net_stas_all_sort=pd.merge(net_stas_all,tissue_type_name_process,left_on='Tissue',right_on='tissue')
net_stas_all_sort=net_stas_all_sort.sort_values('Add number')
net_stas_all_addexp=pd.merge(net_stas_all_sort,exp_tissue,left_on='tissue_name',right_on='Tissue')
net_stas_all_addexp=net_stas_all_addexp[['tissue_name','Tissue-specific mutated genes','Tissue-specific expressed genes','Tissue-specific subnetworks']]
net_stas_all_addexp['Number']=net_stas_all_addexp['Tissue-specific mutated genes']+net_stas_all_addexp['Tissue-specific subnetworks']+net_stas_all_addexp['Tissue-specific expressed genes']
net_stas_all_addexp.rename(columns={'tissue_name':'Tissue'},inplace=True)
net_stas_all_addexp.to_csv('/h/tianyi/TS_datasets_reversion/sci_paper_plot_stas/data_statistic/ts_mut_exp_gene_number/ts_mut_exp_net_num.txt',index=False,sep='\t')
net_stas_all_addexp=net_stas_all_addexp.sort_values('Tissue-specific subnetworks')
ts_mut_exp_net_num = pd.melt(net_stas_all_addexp, id_vars=['Tissue'], 
                  value_vars=['Tissue-specific mutated genes', 'Tissue-specific expressed genes', 'Tissue-specific subnetworks','Number'],
                  var_name='Group', value_name='Total Number')
ts_mut_exp_net_num=ts_mut_exp_net_num.loc[ts_mut_exp_net_num['Group']!='Number',]
custom_palette = {"Tissue-specific mutated genes": "#CC3B4B", "Tissue-specific subnetworks": "#3481B3",'Tissue-specific expressed genes':'#E5C87A'}  # You can customize colors here

ts_mut_exp_net_num['log10num']=np.log10(ts_mut_exp_net_num['Total Number'])
fig,axes=plt.subplots(1,1,figsize=(8.5/2,11/4))
plt.rcParams['axes.linewidth'] = 0.5
axes.spines['top'].set_visible(False)
axes.spines['right'].set_visible(False)
sns.set_style("ticks")
# 绘制双柱状图
sns.barplot(x='Tissue', y='log10num', data=ts_mut_exp_net_num, palette=custom_palette, hue="Group",ax=axes)
# 设置Y轴标签
axes.set_ylabel('Number',fontsize=8, fontname='Arial')
axes.set_xlabel('')
# 设置标题
axes.set_title('',fontsize=8, fontname='Arial')
axes.set_xticklabels(ts_mut_exp_net_num['Tissue'].drop_duplicates().tolist(),fontsize=8, fontname='Arial')
axes.tick_params(labelrotation=90,axis='x')
# 显示图例
plt.legend(prop={'family': 'Arial',  'size': 8})
plt.savefig(f'/h/tianyi/TS_datasets_reversion/sci_paper_plot_stas/data_statistic/ts_mut_exp_gene_number/ts_mut_exp_gene_net_log10_number_primary.pdf',dpi=300,bbox_inches='tight')

plt.savefig(f'/h/tianyi/TS_datasets_reversion/sci_paper_plot_stas/data_statistic/ts_mut_exp_gene_number/ts_mut_exp_gene_net_number_primary.pdf',dpi=300,bbox_inches='tight')


## Figure3B

In [ ]:
library(tidyverse)
library(plotly)
library(ggpubr)
library(ggplot2)
library(ggrepel)
data <- read.csv("degree_betweenness_normalization_all_ts_network.csv")
data <- data |> rename(tissue=Tissue,
                       seed=Subnetwork.seed.genes,
                       gene=Subnetwork.member.genes,
                       gene_type=Gene.reclassification,
                       betweenness=betweenness)

df<- data |> unite("label", tissue, seed, gene, sep = "-")
df$gene <- data$gene

df <- df |> mutate(reduced_gene_type = case_when(
  gene_type %in% c("TSMGs&driver genes", "TSMGs&TSEGs", "TSMGs", "TSMGs&TSEGs&driver genes") ~ "TSMG (Type I)",
  gene_type %in% c("TSEGs", "TSEGs&driver genes") ~ "TSEG (Type II)",
  gene_type == "Driver genes" ~ "COSMIC CGC (Type III)",
  TRUE ~ "Type IV"
))


# 步骤1：按平均值从高到低排序
mean_order <- df %>%
  group_by(reduced_gene_type) %>%
  summarise(mean_value = mean(betweenness)) %>%
  arrange(desc(mean_value)) %>%
  pull(reduced_gene_type)

# 步骤2：应用排序
df$reduced_gene_type <- factor(df$reduced_gene_type, levels = mean_order)

group_colors= c("TSMG (Type I)" = "#3580B2", 
                "TSEG (Type II)" = "#CB3B4B",
                "COSMIC CGC (Type III)" = "#A8C89D",
                "Type IV" = "#DFC379")  # 请根据实际情况设置颜色
p_adjacent <- ggboxplot(df, 
                        x = "reduced_gene_type", 
                        y = "betweenness",
                        fill = "reduced_gene_type",
                        width = 0.45,
                        palette = group_colors,
                        outlier.size = 0.5,
                        #add = "jitter",
                        #add.params = list(
                        #  alpha = 0.5,      # 半透明 (0-1, 0.5表示50%透明)
                        #  size = 0.25,      # 大小 (默认约为3，0.75是默认的0.25倍)
                        #  color = "black"    # 也可以指定颜色
                        #),
                        ggtheme = theme_classic()) +
  ylim(0, NA)+
  labs(x = NULL, 
       y = "Betweenness") +
  theme(legend.position = "none")

# 只比较相邻组对
p_adjacent <- p_adjacent + stat_compare_means(
  method = "wilcox.test",
  comparisons = list(
    c(1, 2),  
    c(2, 3),  
    c(3, 4)
  ),
  label = "p.signif",
  bracket.size = 0.4,
  tip.length = 0.03,
  step.increase = 0.06
)

tiff('/home/yzx46/PC_file/OncoNiche/OncoNiche-Manuscript-26_5_6/compare_type_betweenness_and_degree.tiff',width=6, height=6, units = 'in', res = 300, compression = 'lzw')
p_adjacent+  theme_minimal()+ ##用于手动设置离散型填充颜色的函数
  theme(legend.position = "none",    
        axis.line.y = element_line(size = (0.5/1.07)*0.5),
        axis.line.x = element_line(size = (0.5/1.07)*0.5) ,
        axis.ticks.y = element_line(size = (0.5/1.07)*0.5) ,
        axis.ticks.x = element_line(size = (0.5/1.07)*0.5) ,
        panel.grid=element_blank(),
        axis.text.x = element_text(size = 8, family = "sans", color = "black"),
        axis.text.y = element_text(size = 8, family = "sans", color = "black") # 调整 x 轴刻度线宽度) + # 去除 y 轴刻度线
  )+ # 去除 y 轴刻度线
  ylab('Normalized betweenness')+xlab('')+
  font("xlab",size = 8, family = "sans", color = "black")+
  font("ylab",size = 8, family = "sans", color = "black")
dev.off()



x_90 <- quantile(df$degree, 0.90, na.rm = TRUE)
y_90 <- quantile(df$betweenness, 0.90, na.rm = TRUE)

df$need_label <- with(df, gene == "GRB2"
)


p_ggscatter <- ggscatter(df, x = "degree", y = "betweenness",
               color = "reduced_gene_type",  # 按类型上色
               palette = group_colors,              # 可选配色方案
               size = 0.5,                     # 点的大小
               alpha = 0.7,                  # 点透明度
               ellipse = FALSE,              # 不添加椭圆
               mean.point = FALSE,           # 不显示均值点
               xlab="Normalized degree",
               ylab="Normalized betweenness") +
  geom_hline(yintercept = y_90, linetype = "dashed", color = "gray50") +
  geom_vline(xintercept = x_90, linetype = "dashed", color = "gray50") +
  theme_minimal()+
  theme(legend.position = c(0.1, 0.9),      # 左上角：x=0, y=1
        legend.justification = c(0, 1), # 图例自身的对齐方式：左下角对齐到左上角
        legend.background = element_rect(fill = "white", color = "gray80", size = 0.3),
        legend.margin = margin(5, 5, 5, 5)) +
  labs(color = "gene type")  # 使用 labs() 修改图例标题

# 添加标签（仅标注符合条件的点）
label_data <- subset(df, need_label)
cx <- mean(label_data$degree)
cy <- mean(label_data$betweenness)

label_data <-  label_data %>%
  mutate(
    nudge_x = (degree      - cx) * 0.2,   # 0.5 → 0.2
    nudge_y = (betweenness - cy) * 0.2,
    # 裁剪确保初始位置在合法范围内
    nudge_x = pmax(pmin(degree + nudge_x, 0.95), 0.05) - degree,
    nudge_y = pmax(pmin(betweenness + nudge_y, 0.95), 0.05) - betweenness
  )
p_ggscatter_all=p_ggscatter + geom_text_repel(data = label_data,
                    aes(x = degree, y = betweenness, label = label),
                      nudge_x = label_data$nudge_x,
                 nudge_y = label_data$nudge_y,
                    size = 1.8,           # 标签字号
                    # ── 强制全部显示（关键） ───────────────────────────
                    max.overlaps  = Inf,          # 取消重叠上限，不因重叠丢弃标签
                    min.segment.length = 0,       # 任何距离都画引导线
                    max.iter      = 1e6,          # 增加迭代次数，给算法更多空间排布
                  # ── 排布参数（小图需压缩） ─────────────────────────
                  force         = 150,           # 从 50 → 10，图小时过大的斥力会把标签推出边界
                  force_pull    = 0.1,          # 稍增引力，让标签贴近点
                  box.padding   = 2,          # 从 3 → 0.2，小图下 3 个单位极大
                  point.padding = 0.1,
    # 扩展 repel 算法的活动空间到整个绘图区
    xlim               = c(0.05, 0.95),
    ylim               = c(0.05, 0.95),
                # 引导线允许拉很长
                segment.color = "gray60",
                segment.size       = 0.2,
                segment.alpha      = 0.5,
                segment.curvature  = 0.1,        # 轻微弯曲，视觉上更易追踪
                    )  
pdf('/home/yzx46/PC_file/OncoNiche/OncoNiche-Manuscript-26_5_6/betweenness_and_degree.pdf',width=11/3, height=11/3 )
p_ggscatter_all+  theme_minimal()+
  theme(legend.position = "none",    
        axis.line.y = element_line(size = (0.5/1.07)*0.5),
        axis.line.x = element_line(size = (0.5/1.07)*0.5) ,
        axis.ticks.y = element_line(size = (0.5/1.07)*0.5) ,
        axis.ticks.x = element_line(size = (0.5/1.07)*0.5) ,
        panel.grid=element_blank(),
        axis.text.x = element_text(size = 8, family = "sans", color = "black"),
        axis.text.y = element_text(size = 8, family = "sans", color = "black") # 调整 x 轴刻度线宽度) + # 去除 y 轴刻度线
  )+ # 去除 y 轴刻度线
  xlab('Normalized degree')+ylab('Normalized betweenness')+
  font("xlab",size = 8, family = "sans", color = "black")+
  font("ylab",size = 8, family = "sans", color = "black")+
scale_x_continuous(limits = c(0, 1), expand = expansion(mult = c(0, 0.02))) +
scale_y_continuous(limits = c(0, 1), expand = expansion(mult = c(0, 0.02)))
dev.off()



pdf('/home/yzx46/PC_file/OncoNiche/OncoNiche-Manuscript-26_5_6/betweenness_and_degree_legend.pdf',width=11/3 , height=11/3 )
p_ggscatter_all+  theme_minimal()+
  theme(
        axis.line.y = element_line(size = (0.5/1.07)*0.5),
        axis.line.x = element_line(size = (0.5/1.07)*0.5) ,
        axis.ticks.y = element_line(size = (0.5/1.07)*0.5) ,
        axis.ticks.x = element_line(size = (0.5/1.07)*0.5) ,
        panel.grid=element_blank(),
        axis.text.x = element_text(size = 8, family = "sans", color = "black"),
        axis.text.y = element_text(size = 8, family = "sans", color = "black") # 调整 x 轴刻度线宽度) + # 去除 y 轴刻度线
  )+ # 去除 y 轴刻度线
  xlab('Normalized degree')+ylab('Normalized betweenness')+
  font("xlab",size = 8, family = "sans", color = "black")+
  font("ylab",size = 8, family = "sans", color = "black")+
scale_x_continuous(limits = c(0, 1), expand = expansion(mult = c(0, 0.02))) +
scale_y_continuous(limits = c(0, 1), expand = expansion(mult = c(0, 0.02)))
dev.off()

## Figure3C

In [ ]:
## specific pathway
library(clusterProfiler)
library(ggplot2)
specific_pathway_p_GR_all=data.frame(read.table('specific_pathway_p_GR_all.txt',sep='\t',header=TRUE))
tissue_color=data.frame(read.table('tissue_color.txt',sep='\t',header=TRUE))
specific_pathway_p_GR_all=merge(specific_pathway_p_GR_all,tissue_color,by='tissue')
specific_pathway_p_GR_all$pathway_id_plot <- factor(specific_pathway_p_GR_all$pathway_id_pro, levels = unique(specific_pathway_p_GR_all$pathway_id_pro))
specific_pathway_p_GR_all$Tissue_gene=paste(specific_pathway_p_GR_all$tissue_name,specific_pathway_p_GR_all$seed_gene,sep=' ')
specific_pathway_p_GR_all_sort=specific_pathway_p_GR_all[order(specific_pathway_p_GR_all$Tissue_gene),]

pdf('specific_pathway_plot_v3.pdf',height = 3.3, width =4.3)
ggplot(specific_pathway_p_GR_all_sort[1:15,], aes(Tissue_gene,pathway_id_plot ), showCategory=1) +
  geom_point(aes(color=FDR_BH , size=GeneRatio))+theme(axis.text.x = element_text(angle = 60, hjust = 1, size = 8),
                                                       axis.line.y = element_line(size = (0.5/1.07)*0.5),
                                                       axis.line.x = element_line(size = (0.5/1.07)*0.5) ,
                                                       axis.ticks.y = element_line(size = (0.5/1.07)*0.5) ,
                                                       axis.ticks.x = element_line(size = (0.5/1.07)*0.5) ,
                                                       axis.text.y = element_text(size = 8)  ,
                                                       axis.title.x = element_blank()  ,axis.title.y = element_blank(),
                                                       panel.border = element_rect(color = "black", fill = NA, size = 0.5),
                                                       panel.background = element_blank())+ 
  scale_color_gradient(low = "#E3475A", high = "#626EB3",limits = c(0, 0.05))+scale_size_continuous(range = c(0.5, 3))+
  theme(legend.position = "none") +theme(panel.grid.minor = element_blank(), panel.grid.major = element_blank())
dev.off()

pdf('specific_pathway_plot_legend.pdf',height = 4, width = 8)
ggplot(specific_pathway_p_GR_all_sort, aes(Tissue_gene,pathway_id_plot ), showCategory=1) +
  geom_point(aes(color=FDR_BH , size=GeneRatio))+theme(axis.text.x = element_text(angle = 90, hjust = 1, size = 8),
                                                       axis.text.y = element_text(size = 8)  ,axis.title.x = element_blank()  ,axis.title.y = element_blank(),
                                                       panel.border = element_rect(color = "black", fill = NA, size = 0.5),
                                                       panel.background = element_blank())+ 
  scale_color_gradient(low = "#E3475A", high = "#626EB3",limits = c(0, 0.05))+scale_size_continuous(range = c(0.5, 3))+
  theme(panel.grid.minor = element_blank(), panel.grid.major = element_blank())+theme(legend.position = "bottom")
dev.off()

## figure3D

In [ ]:
###apcluster and share pathway

options(repos = "https://mirrors.tuna.tsinghua.edu.cn/CRAN/")
install.packages("apcluster") # The downloaded source packages are in /tmp/Rtmp4IZp7W/downloaded_packages
install.packages("yarrr")
library(apcluster)
library(ComplexHeatmap)
library(colorRamp2)
library(yarrr)
library(ggplot2)
library(readxl)
setwd('/home/yzx46/PC_file/OncoNiche/OncoNiche-Manuscript-26_4_3/pathway_apcluster')
pathway_jaccard=data.frame(read.table('jaccard_matrix.txt',header = T,row.names=1,stringsAsFactors = FALSE,
                                  check.names = FALSE,sep='\t'))
pathway_jaccard_matrix=as.matrix(pathway_jaccard)
# 使用负欧式距离作为相似度（越大表示越相似）
sim <- negDistMat(r=2, pathway_jaccard_matrix)
# 运行标准的 affinity propagation 聚类
ap_result <- apcluster(sim)
# 查看结果

pathway_cluster_id=data.frame(pathway = rownames(pathway_jaccard_matrix),
                              cluster_id =  labels(ap_result, type = "enum"))
exemplars_path=data.frame(pathway_exemplars=names(ap_result@exemplars),
                          cluster_id=c(1:6))
exemplars_path_cluster=merge(pathway_cluster_id,exemplars_path,by='cluster_id')
dim(pathway_jaccard_matrix[exemplars_path_cluster$pathway,])
colnames(pathway_jaccard_matrix)=gsub('\\.',' ',colnames(pathway_jaccard_matrix))
ordered_mat <- pathway_jaccard_matrix[exemplars_path_cluster$pathway, exemplars_path_cluster$pathway]

library(stringr)
matrix_colnames=gsub('REACTOME ','',colnames(ordered_mat))
matrix_colnames_title=str_to_sentence(tolower(as.character(matrix_colnames)))
matrix_colnames_title_pro=gsub('Diseases of signal transduction by growth factor receptors and second messengers','Diseases of signal...messengers(R-HSA-5663202)',matrix_colnames_title)
matrix_colnames_title_pro=gsub('Signaling by type 1 insulin like growth factor 1 receptor igf1r','Signaling by...receptor igf1r(R-HSA-2404192)',matrix_colnames_title_pro)
colnames(ordered_mat) = matrix_colnames_title_pro

pathway_color <- list(
  Exemplars = c("Diseases of signal...messengers(R-HSA-5663202)" = "#D83B4E", 
                "Cytokine signaling in immune system" = "#3488BE",
                'Infectious disease'='#B1D5A4',
                'Signaling by nuclear receptors'='#eccd7a',
                'Insulin receptor signalling cascade'='#9FBDE4',
                'Signaling by egfr in cancer'='#EF9BBB'))
exemplars_path_cluster_name=exemplars_path_cluster$pathway_exemplars
exemplars_path_cluster_name1=gsub('REACTOME ','',exemplars_path_cluster_name)
exemplars_path_cluster_name2=str_to_sentence(tolower(as.character(exemplars_path_cluster_name1)))
exemplars_path_cluster_name3=gsub('Diseases of signal transduction by growth factor receptors and second messengers','Diseases of signal...messengers(R-HSA-5663202)',exemplars_path_cluster_name2)
exemplars_path_cluster_name4=gsub('Signaling by type 1 insulin like growth factor 1 receptor igf1r','Signaling by...receptor igf1r(R-HSA-2404192)',exemplars_path_cluster_name3)

pathway_color_row_anno <- HeatmapAnnotation(  Exemplars = exemplars_path_cluster_name4,
                                             col = pathway_color)

pdf("share_pathway_heatmap.pdf", width = 8, height = 4.5)
heatmap_seed_plot=Heatmap(ordered_mat,
                          top_annotation = pathway_color_row_anno,
                          name = "Jaccard index",
                          col = colorRamp2(c(min(ordered_mat), max(ordered_mat)), c( "white", "red")),
                          show_row_names = FALSE,
                          show_column_names = TRUE,
                          cluster_rows =F ,             
                          cluster_columns =F ,         
                          cluster_row_slices = FALSE,       
                          cluster_column_slices = FALSE ,
                          show_row_dend = FALSE,show_column_dend = FALSE,
                          row_names_gp = gpar(fontfamily = "sans", fontsize = 8, col = "black"),
                          column_names_gp = gpar(fontfamily = "sans", fontsize = 8, col = "black"))
draw(
  heatmap_seed_plot,
  show_annotation_legend = FALSE,   # 关闭 annotation 图例
  show_heatmap_legend = TRUE        # 保留 heatmap color bar
)
dev.off()

pdf("share_pathway_heatmap_nolegend.pdf", width = 8, height = 4.5)
heatmap_seed_plot=Heatmap(ordered_mat,
                          name = "Jaccard index",
                          col = colorRamp2(c(min(ordered_mat), max(ordered_mat)), c( "white", "red")),
                          show_row_names = FALSE,
                          show_column_names = TRUE,
                          cluster_rows =F ,             
                          cluster_columns =F ,         
                          cluster_row_slices = FALSE,       
                          cluster_column_slices = FALSE ,
                          show_row_dend = FALSE,show_column_dend = FALSE,
                          row_names_gp = gpar(fontfamily = "sans", fontsize = 8, col = "black"),
                          column_names_gp = gpar(fontfamily = "sans", fontsize = 8, col = "black"))
draw(heatmap_seed_plot)
dev.off()